# Problem 020:  Factorial Digit Sum

> $n!$ means $n \times (n - 1) \times \cdots \times 3 \times 2 \times 1$.
>
> For example, $10! = 10 \times 9 \times \cdots \times 3 \times 2 \times 1 = 3628800$, <br>
> and the sum of the digits in the number $10!$ is $3 + 6 + 2 + 8 + 8 + 0 + 0 = 27$.
>
> Find the sum of the digits in the number $100!$.

## Using `factorial`

With no worries about overflowing `int`'s and a built in `factorial` function in `math`, the Pythonic solution can be a one-liner.

In [74]:
%%time

from math import factorial

def p020_python(N: int) -> int:
    return sum(map(int, str(factorial(N))))
    
N = 100
ans = p020_python(N)

print(ans)


648
CPU times: user 236 μs, sys: 16 μs, total: 252 μs
Wall time: 244 μs


## Non-Pythonic Solution: Using an array of digits

Without any concern about integer overflow, the method above is as good as anything.  But, if we were to be concerned by integer overflows, then the value of the factorial would need to be calculated in an array of digits.

In this case, a few modifications can be used.  First, there are going to be $24$ trailing $0$'s at the end of $100!$, which are unnecessary to carry around.  Thus, when calculating the factorial factors of $10$ can be divided out.  The way I have implemented this below is by counting the number of factors of $5$ in the factorial in the variable `n5`, always dividing out factors of $5$ when multiplying the next number, and dividing out the first `n5` factors of $2$ seen.

I prefer to allocate the required space of an array as opposed to dynamically appending values to the end.  For this purpose, Stirling's approximation is a little frustrating to use since it always underapproximates the length:
$$\ln n! \simeq \frac{1}{2}\ln (2\pi n) + n \ln n - n$$
By avoiding the factors of 10 in the factorial, using this approximation for the length of the array suffices, even if it is a bit larger than necessary.


In [102]:
%%time

from math import log, pi
from warnings import warn

def multiply_digit_array_by_int(a: list[int], x: int, ndig: int) -> int:
    carry = 0
    for i in range(ndig):
        carry += x * a[i]
        a[i] = carry % 10
        carry //= 10
    while carry != 0 and ndig != len(a):
        a[ndig] = carry % 10
        carry //= 10
        ndig += 1
    if carry != 0:
        warn("Array not large enough to complete multiplication.")
    return ndig

def p020_digit_array(N: int) -> int:
    lmax = int((0.5 * log(2 * pi * N) + N * log(N) - N) / log(10)) + 1
    a = [None] * lmax

    # calculate the number of factors of $10$ found in $N!$
    n5 = 0
    ip = N // 5
    while ip:
        n5 += ip
        ip //= 5
        
    N += 1
    ndig = 1
    a[0] = 2
    for i in range(3, N):
        ip = i
        while ip % 5 == 0:
            ip //= 5
        while n5 and ip % 2 == 0:
            ip //= 2
            n5 -= 1
        ndig = multiply_digit_array_by_int(a, ip, ndig)
    return sum(a[:ndig])

N = 100
ans = p020_digit_array(N)
print(ans)

648
CPU times: user 1.42 ms, sys: 0 ns, total: 1.42 ms
Wall time: 1.32 ms
